# 03 · Score dump

Runs every trained checkpoint once over ASVspoof 2019 LA dev and eval and saves, per utterance, the spoof logit and the pooled embedding (plus wav2vec2 frame attention). All metrics, fusion and charts are computed from these files on CPU in notebook 04, so no GPU run is ever repeated for evaluation.

Inputs: the ASVspoof 2019 LA dataset, notebook 00 output, `audio_utils.py`, `rawboost.py` and the checkpoints from notebooks 01 and 02 (any file names; each is identified by its contents). No Internet is needed.

Inference follows the training notebooks exactly: the first 4 s of each clip, zero-padded; the ResNet18 input is normalised log-mel + Δ + ΔΔ with the checkpoint's train statistics; the wav2vec2 input is normalised over its real samples and then padded. Audio is read from the original FLAC files for both splits, so the dev check below also confirms that FLAC decoding matches notebook 00's cache.

| Output | Contents |
|---|---|
| `scores/{model}_seed{s}_{split}.csv` | utt_id, split, label, attack, logit, model, checkpoint and utils hashes |
| `embeddings/{model}_seed{s}_{split}.npz` | `utt_id` and float16 pooled embeddings (512-d ResNet18, 768-d wav2vec2) in score-file order |
| `attention/wav2vec2_seed{s}_{split}.npz` | `utt_id` and float16 frame attention weights |
| `score_manifest.json` | Checkpoint hashes, row counts, dev-EER reproduction, library versions |

In [1]:
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18
from transformers import Wav2Vec2Config, Wav2Vec2Model

INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
EXPECTED_MODELS = ("resnet18", "wav2vec2")
SPLITS = ("dev", "eval")
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()
N_GPU = torch.cuda.device_count()
DEVICES = {"resnet18": torch.device("cuda:0" if N_GPU else "cpu"),
           "wav2vec2": torch.device(f"cuda:{min(1, N_GPU - 1)}" if N_GPU else "cpu")}
AMP = N_GPU > 0
print(f"torch {torch.__version__}, transformers {transformers.__version__}, "
      f"{[torch.cuda.get_device_name(i) for i in range(N_GPU)] or 'CPU only'}", flush=True)

torch 2.11.0+cu128, transformers 5.16.1, ['Tesla T4', 'Tesla T4']


In [2]:
def walk_inputs(max_depth=6):
    for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
        depth = os.path.relpath(dirpath, INPUT_ROOT).count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if d not in ("flac", "cache"))
        yield dirpath, dirnames, filenames


def find_dir(required=(), subdir=None):
    hits = {os.path.realpath(d) for d, sub, files in walk_inputs()
            if set(required) <= set(files) and (subdir is None or subdir in sub)}
    if len(hits) != 1:
        raise FileNotFoundError(f"expected one directory with {required or subdir} under {INPUT_ROOT}, "
                                f"found {sorted(hits)}")
    return hits.pop()


for module_file in ("audio_utils.py", "rawboost.py"):
    sys.path.insert(0, find_dir([module_file]))
import audio_utils as U  # noqa: E402

U._self_test()
LA_ROOT = find_dir(subdir="ASVspoof2019_LA_cm_protocols")
PREP_DIR = find_dir(["manifest.csv", "feature_stats.json", "provenance.json"])
with open(os.path.join(PREP_DIR, "provenance.json")) as fh:
    prep = json.load(fh)
if U.file_sha256(os.path.join(PREP_DIR, "manifest.csv")) != prep["manifest_sha256"]:
    raise AssertionError("manifest.csv does not match provenance.json")
for split, name in U.PROTOCOL_FILES.items():
    if U.file_sha256(os.path.join(LA_ROOT, "ASVspoof2019_LA_cm_protocols", name)) != prep["protocol_sha256"][split]:
        raise AssertionError(f"{split} protocol differs from the one used in notebook 00")
manifest = pd.read_csv(os.path.join(PREP_DIR, "manifest.csv"))
with open(os.path.join(PREP_DIR, "feature_stats.json")) as fh:
    prep_stats = json.load(fh)
print(LA_ROOT, PREP_DIR, sep="\n", flush=True)

audio_utils 1.0.1 self-test passed (sha 658e72bce982)
/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data
/kaggle/input/notebooks/rudranshiverma/00-data-prep


## Checkpoints

Every `.pt` / `.pth` file under the inputs is opened and identified by its `model` field. Each must be the final checkpoint written by the last cell of notebook 01 or 02, carry a dev-EER threshold, and come from the same notebook 00 manifest.

In [3]:
REQUIRED = {
    "resnet18": ("model_state_dict", "seed", "dev_eer", "threshold_logit", "threshold_source", "feature_stats",
                 "prep_manifest_sha256"),
    "wav2vec2": ("model_state_dict", "seed", "dev_eer", "threshold_logit", "threshold_source", "encoder_config",
                 "prep_manifest_sha256"),
}
checkpoints = {}
for d, _, files in walk_inputs():
    for f in sorted(files):
        if not f.endswith((".pt", ".pth")):
            continue
        path = os.path.join(d, f)
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        name = ckpt.get("model") if isinstance(ckpt, dict) else None
        if name not in REQUIRED:
            raise ValueError(f"{path}: not a final checkpoint from notebook 01 or 02 (did its last cell run?)")
        missing = [k for k in REQUIRED[name] if k not in ckpt]
        if missing:
            raise ValueError(f"{path}: missing {missing}")
        if ckpt["threshold_source"] != "dev_eer":
            raise ValueError(f"{path}: threshold does not come from dev")
        if ckpt["prep_manifest_sha256"] != prep["manifest_sha256"]:
            raise ValueError(f"{path}: trained on a different notebook 00 manifest")
        if name == "resnet18" and ckpt["feature_stats"] != prep_stats:
            raise ValueError(f"{path}: feature statistics differ from notebook 00")
        key = (name, ckpt["seed"])
        if key in checkpoints:
            raise ValueError(f"two checkpoints for {key}: {checkpoints[key]['path']} and {path}")
        checkpoints[key] = {"ckpt": ckpt, "path": path, "sha256": U.file_sha256(path)}
found = sorted({name for name, _ in checkpoints})
if found != sorted(EXPECTED_MODELS):
    raise FileNotFoundError(f"expected checkpoints for {EXPECTED_MODELS}, found {found}")
pd.DataFrame([{"model": n, "seed": s, "epoch": c["ckpt"].get("epoch"), "dev_eer": c["ckpt"]["dev_eer"],
               "file": os.path.relpath(c["path"], INPUT_ROOT)} for (n, s), c in checkpoints.items()])

,model,seed,epoch,dev_eer,file
0,resnet18,0,7,0.025605,notebooks/rudranshiverma/01-resnet18/resnet18_...
1,wav2vec2,0,6,0.001172,notebooks/rudranshiverma/02-wav2vec2/wav2vec2_...


## Models

Architectures are identical to notebooks 01 and 02; `load_state_dict(strict=True)` rejects any mismatch, and the dev check below rejects any difference in behaviour.

In [4]:
class FP32Head(nn.Sequential):
    def forward(self, x):
        with torch.autocast(x.device.type, enabled=False):
            return super().forward(x.float())


class ResNetScorer(nn.Module):
    def __init__(self, ckpt):
        super().__init__()
        self.net = resnet18(weights=None)
        self.net.fc = FP32Head(nn.BatchNorm1d(512), nn.Dropout(0.3), nn.Linear(512, 1))
        self.net.load_state_dict(ckpt["model_state_dict"], strict=True)
        self.features = U.LogMelDelta(U.FeatureStats.from_dict(ckpt["feature_stats"]))

    def forward(self, raw, _):
        with torch.autocast(raw.device.type, enabled=False):
            x = self.features(raw)
        captured = {}
        hook = self.net.fc.register_forward_hook(lambda m, inp, out: captured.update(emb=inp[0]))
        logit = self.net(x).squeeze(1)
        hook.remove()
        return logit, captured["emb"], None


class W2VClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        dim = encoder.config.hidden_size
        self.layer_logits = nn.Parameter(torch.zeros(encoder.config.num_hidden_layers + 1))
        self.attention = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Sequential(nn.Linear(dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
                                        nn.Linear(256, 1))

    def embed(self, wave):
        hidden = self.encoder(wave, output_hidden_states=True).hidden_states
        with torch.autocast(wave.device.type, enabled=False):
            stacked = torch.stack([h.float() for h in hidden], dim=0)
            mixed = (torch.softmax(self.layer_logits, dim=0)[:, None, None, None] * stacked).sum(dim=0)
            attn = torch.softmax(self.attention(mixed), dim=1)
            pooled = (attn * mixed).sum(dim=1)
        return pooled, attn.squeeze(-1)


class W2VScorer(nn.Module):
    def __init__(self, ckpt):
        super().__init__()
        cfg = {k: v for k, v in ckpt["encoder_config"].items() if k != "transformers_version"}
        self.net = W2VClassifier(Wav2Vec2Model(Wav2Vec2Config(**cfg)))
        self.net.load_state_dict(ckpt["model_state_dict"], strict=True)

    def forward(self, _, wave):
        pooled, attn = self.net.embed(wave)
        with torch.autocast(wave.device.type, enabled=False):
            logit = self.net.classifier(pooled).squeeze(1)
        return logit, pooled, attn


scorers = {}
for (name, seed), c in checkpoints.items():
    scorer = (ResNetScorer if name == "resnet18" else W2VScorer)(c["ckpt"])
    scorers[(name, seed)] = scorer.to(DEVICES[name]).eval()
print(f"loaded {sorted(scorers)}", flush=True)

loaded [('resnet18', 0), ('wav2vec2', 0)]


## Scoring

Each clip is decoded once and fed to every model.

In [5]:
def normalise_wave(x):
    return (x - x.mean()) / np.sqrt(x.var() + 1e-7)


class FlacClips(Dataset):
    def __init__(self, df):
        self.paths = [os.path.join(LA_ROOT, p) for p in df["rel_path"]]
        self.lengths = df["n_samples"].to_numpy()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        x = U.int16_to_float32(U.read_audio_int16(self.paths[i]))
        if x.size != self.lengths[i]:
            raise AssertionError(f"{self.paths[i]}: {x.size} samples, manifest says {self.lengths[i]}")
        head = x[:U.MAX_SAMPLES]
        return U.fit_head(torch.from_numpy(head)), U.fit_head(torch.from_numpy(normalise_wave(head)))


@torch.no_grad()
def score_split(df):
    loader = DataLoader(FlacClips(df), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=True)
    out = {k: {"logit": [], "emb": [], "attn": []} for k in scorers}
    n_batches, log_every, t0 = len(loader), max(1, len(loader) // 10), time.time()
    for step, (raw, wave) in enumerate(loader, start=1):
        for key, scorer in scorers.items():
            dev = DEVICES[key[0]]
            with torch.autocast(dev.type, dtype=torch.float16, enabled=AMP):
                logit, emb, attn = scorer(raw.to(dev, non_blocking=True), wave.to(dev, non_blocking=True))
            out[key]["logit"].append(logit.float().cpu())
            out[key]["emb"].append(emb.half().cpu())
            if attn is not None:
                out[key]["attn"].append(attn.half().cpu())
        if step % log_every == 0:
            elapsed = time.time() - t0
            print(f"  batch {step}/{n_batches}  {elapsed:.0f} s, ~{elapsed / step * (n_batches - step):.0f} s left",
                  flush=True)
    return {k: {f: torch.cat(v).numpy() for f, v in d.items() if v} for k, d in out.items()}


for sub in ("scores", "embeddings", "attention"):
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)
results = {}
for split in SPLITS:
    df = manifest[manifest["split"] == split].reset_index(drop=True)
    print(f"{split}: {len(df)} clips", flush=True)
    results[split] = (df, score_split(df))

dev: 24844 clips
  batch 38/389  28 s, ~261 s left
  batch 76/389  51 s, ~209 s left
  batch 114/389  75 s, ~181 s left
  batch 152/389  98 s, ~153 s left
  batch 190/389  121 s, ~127 s left
  batch 228/389  145 s, ~102 s left
  batch 266/389  168 s, ~78 s left
  batch 304/389  191 s, ~53 s left
  batch 342/389  215 s, ~30 s left
  batch 380/389  238 s, ~6 s left
eval: 71237 clips
  batch 111/1114  71 s, ~639 s left
  batch 222/1114  139 s, ~557 s left
  batch 333/1114  206 s, ~484 s left
  batch 444/1114  274 s, ~414 s left
  batch 555/1114  342 s, ~345 s left
  batch 666/1114  410 s, ~276 s left
  batch 777/1114  478 s, ~207 s left
  batch 888/1114  546 s, ~139 s left
  batch 999/1114  614 s, ~71 s left
  batch 1110/1114  682 s, ~2 s left


## Checks and outputs

Re-scored dev must reproduce each checkpoint's recorded dev EER, and each saved embedding must reproduce its logit through the model's own head, so the embeddings are exactly what the classifier saw. Eval EER is shown only as a sanity check; the full evaluation is in notebook 04.

In [6]:
summary = []
for (name, seed), c in checkpoints.items():
    head = scorers[(name, seed)].net.fc if name == "resnet18" else scorers[(name, seed)].net.classifier
    for split in SPLITS:
        df, res = results[split]
        r = res[(name, seed)]
        logits, labels = r["logit"], df["label"].to_numpy()
        if not np.isfinite(logits).all():
            raise AssertionError(f"{name} {split}: non-finite logits")
        idx = np.random.default_rng(0).choice(len(df), size=min(256, len(df)), replace=False)
        with torch.no_grad():
            dev = DEVICES[name]
            again = head(torch.from_numpy(r["emb"][idx].astype(np.float32)).to(dev)).squeeze(1).float().cpu().numpy()
        if np.max(np.abs(again - logits[idx])) > 0.05 * max(1.0, np.abs(logits[idx]).max()):
            raise AssertionError(f"{name} {split}: embeddings do not reproduce logits")
        value, _ = U.eer(labels, logits)
        step = 1.0 / min((labels == 0).sum(), (labels == 1).sum())
        if split == "dev" and abs(value - c["ckpt"]["dev_eer"]) > step:
            raise AssertionError(f"{name}: dev EER {value:.5f} != training-time {c['ckpt']['dev_eer']:.5f}")
        tag = f"{name}_seed{seed}_{split}"
        U.write_scores(df.assign(logit=logits), os.path.join(WORK_DIR, "scores", f"{tag}.csv"), f"{name}_seed{seed}",
                       c["sha256"])
        np.savez(os.path.join(WORK_DIR, "embeddings", f"{tag}.npz"), utt_id=df["utt_id"].to_numpy(), emb=r["emb"])
        if "attn" in r:
            np.savez(os.path.join(WORK_DIR, "attention", f"{tag}.npz"), utt_id=df["utt_id"].to_numpy(),
                     attn=r["attn"])
        summary.append({"model": name, "seed": seed, "split": split, "clips": len(df), "eer": value,
                        "checkpoint_dev_eer": c["ckpt"]["dev_eer"] if split == "dev" else None})
summary = pd.DataFrame(summary)
summary

,model,seed,split,clips,eer,checkpoint_dev_eer
0,resnet18,0,dev,24844,0.025846,0.025605
1,resnet18,0,eval,71237,0.063086,NaN
2,wav2vec2,0,dev,24844,0.001172,0.001172
3,wav2vec2,0,eval,71237,0.010601,NaN


In [7]:
for path in sorted(os.listdir(os.path.join(WORK_DIR, "scores"))):
    back = U.read_scores(os.path.join(WORK_DIR, "scores", path), manifest)
    if len(back) != (manifest["split"] == back["split"].iloc[0]).sum():
        raise AssertionError(f"{path}: wrong row count")

score_manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "utils_version": U.UTILS_VERSION,
    "utils_sha256": U.utils_sha256(),
    "prep_manifest_sha256": prep["manifest_sha256"],
    "checkpoints": {f"{n}_seed{s}": {"file": os.path.relpath(c["path"], INPUT_ROOT), "sha256": c["sha256"],
                                     "epoch": c["ckpt"].get("epoch"), "dev_eer": c["ckpt"]["dev_eer"],
                                     "threshold_logit": c["ckpt"]["threshold_logit"]}
                    for (n, s), c in checkpoints.items()},
    "summary": summary.to_dict(orient="records"),
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
}
with open(os.path.join(WORK_DIR, "score_manifest.json"), "w") as fh:
    json.dump(score_manifest, fh, indent=2)
print(json.dumps(score_manifest["checkpoints"], indent=2))

{
  "resnet18_seed0": {
    "file": "notebooks/rudranshiverma/01-resnet18/resnet18_seed0.pt",
    "sha256": "89beeb29853d571f0e5ab973b914661f8bc8deb7aaa2c481a7f95ae8897c8ffe",
    "epoch": 7,
    "dev_eer": 0.02560494057687661,
    "threshold_logit": 2.4378979206085205
  },
  "wav2vec2_seed0": {
    "file": "notebooks/rudranshiverma/02-wav2vec2/wav2vec2_seed0.pt",
    "sha256": "58dd55996a0c759c8eb97b64bd0f6ef35b2e41090d97fafdcca9521374576212",
    "epoch": 6,
    "dev_eer": 0.001171761244035579,
    "threshold_logit": 4.28126859664917
  }
}
